# AutoEncoder Baseline — Training on Colab T4

Everything runs **locally on the Colab instance** (no Drive mount).

**Pipeline :**
1. Check GPU
2. Clone repo
3. Generate simple pendulum dataset
4. Train AutoEncoder baseline (encoder + predictor + decoder, supervision pixel-space)
5. Plot loss curves
6. Download best checkpoint

**Différence clé vs LeWorldModel (JEPA) :**

| | AutoEncoder | LeWorldModel |
|---|---|---|
| Supervision | MSE reconstruction pixel | Cosine distance espace latent |
| Décodeur pendant training | ✓ oui (joint) | ✗ non (séparé) |
| Target encoder EMA | ✗ non | ✓ oui |
| SIGReg | ✗ non | ✓ oui |

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. GPU check

In [ ]:
import subprocess, torch

print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
print(f'PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  device: {torch.cuda.get_device_name(0)}')
assert torch.cuda.is_available(), 'No GPU found — switch to T4 runtime first.'

## 2. Clone repo

In [ ]:
import os

REPO_URL = 'https://github.com/JulesV19/double-pendule-wolrdmodel.git'
REPO_DIR = '/content/WorldModel'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

## 3. Install dependencies

In [ ]:
!pip install -q Pillow matplotlib numpy
print('Dependencies ready.')

## 4. Generate dataset

Generates **2 000 trajectories × 500 frames** (64×64 px) of a simple pendulum.  
Each trajectory has random initial angle **and** random initial velocity.  
~5-10 min on Colab CPU.

During training, a random **window of 100 frames** is sampled from each 500-frame
trajectory at every epoch — giving 5× more temporal diversity than a fixed dataset.

In [ ]:
from pathlib import Path

DATASET_DIR = 'dataset/pendulum'
N_TRAJ      = 2000
N_FRAMES    = 500
IMG_SIZE    = 64

existing = len(list(Path(DATASET_DIR).glob('traj_*.npz'))) if Path(DATASET_DIR).exists() else 0

if existing >= N_TRAJ:
    print(f'Dataset already present ({existing} trajectories) — skipping generation.')
else:
    print(f'Generating {N_TRAJ} trajectories × {N_FRAMES} frames…')
    from generate_dataset import generate_dataset
    generate_dataset(
        n_trajectories=N_TRAJ,
        n_frames=N_FRAMES,
        img_size=IMG_SIZE,
        dt=0.05,
        output_dir=DATASET_DIR,
        seed=42,
    )

### Quick dataset preview

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample = np.load(f'{DATASET_DIR}/traj_0000.npz')
frames = sample['frames']   # (T, H, W, 3)

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
fig.patch.set_facecolor('#111')
for ax, idx in zip(axes, np.linspace(0, len(frames)-1, 8, dtype=int)):
    ax.imshow(frames[idx])
    ax.axis('off')
    ax.set_title(f't={idx}', color='white', fontsize=8)
plt.suptitle('Trajectory 0 — 8 frames', color='white')
plt.tight_layout()
plt.show()

## 5. Training

| Hyperparameter | Value | Note |
|---|---|---|
| `embed_dim` | 128 | latent dimension — identique à LeWorldModel |
| `hidden_dim` | 512 | MLP transition FFN — identique |
| `rollout_k` | 5 | rollout k=1…5 steps, supervision pixel à chaque step |
| `seq_len` | 100 | window sampled randomly from 500-frame trajectories |
| `epochs` | 50 | |
| `batch_size` | 32 | fits T4 16 GB |
| `lr` | 1e-4 | AdamW + warmup 5ep + cosine |

Pas de SIGReg, pas de target encoder EMA — le décodeur est dans la boucle de training.

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
EMBED_DIM    = 128
HIDDEN_DIM   = 512
ROLLOUT_K    = 5     # rollout k=1…5, supervision MSE pixel à chaque step
SEQ_LEN      = 100   # fenêtre tirée aléatoirement dans chaque trajectoire
EPOCHS       = 50
BATCH_SIZE   = 32
LR           = 1e-4
WEIGHT_DECAY = 0.05
NUM_WORKERS  = 2     # Colab has 2 CPU cores
CKPT_DIR     = 'checkpoints'
VIS_DIR      = 'visuals'
RESUME       = None  # set to 'checkpoints/ae_best.pt' to resume

In [ ]:
import time, math
from pathlib import Path

import torch
import torch.optim as optim
import matplotlib.pyplot as plt

from models.ae import AutoEncoder
from dataset import make_seq_dataloaders

device = torch.device('cuda')
print(f'Training on: {torch.cuda.get_device_name(0)}')

# ── Dataloaders ────────────────────────────────────────────────────────────────
train_loader, val_loader = make_seq_dataloaders(
    dataset_dir=DATASET_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seq_len=SEQ_LEN,
)
print(f'Train: {len(train_loader)} batches  |  Val: {len(val_loader)} batches')

# ── Model ──────────────────────────────────────────────────────────────────────
model = AutoEncoder(
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    rollout_k=ROLLOUT_K,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')

# ── Optimizer & scheduler ──────────────────────────────────────────────────────
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    warmup = 5
    if epoch < warmup:
        return (epoch + 1) / warmup
    t = (epoch - warmup) / max(1, EPOCHS - warmup)
    return 0.5 * (1 + math.cos(math.pi * t))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Evaluate ───────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total = 0.0
    for frames, _ in loader:
        m = model(frames.to(device, non_blocking=True))
        total += m['loss'].item()
    return total / len(loader)

# ── Resume ─────────────────────────────────────────────────────────────────────
start_epoch = 1
if RESUME and Path(RESUME).exists():
    ckpt = torch.load(RESUME, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda, last_epoch=ckpt['epoch'])
    print(f'Resumed from epoch {ckpt["epoch"]}')

Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(VIS_DIR).mkdir(parents=True, exist_ok=True)

torch.cuda.reset_peak_memory_stats(device)

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────

history  = {'train': [], 'val': [], 'epoch_time': []}
best_val = float('inf')

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t0    = time.time()
    total = 0.0

    for frames, _ in train_loader:
        frames = frames.to(device, non_blocking=True)
        optimizer.zero_grad()
        m = model(frames)
        m['loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total += m['loss'].item()

    scheduler.step()
    elapsed    = time.time() - t0
    n          = len(train_loader)
    train_loss = total / n
    val_loss   = evaluate(model, val_loader)

    history['train'].append(train_loss)
    history['val'].append(val_loss)
    history['epoch_time'].append(elapsed)

    improved = val_loss < best_val
    if improved:
        best_val = val_loss
        torch.save({
            'epoch':    epoch,
            'model':    model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'val_loss': best_val,
            'args': {
                'embed_dim':  EMBED_DIM,
                'hidden_dim': HIDDEN_DIM,
                'rollout_k':  ROLLOUT_K,
            },
        }, f'{CKPT_DIR}/ae_best.pt')

    lr_now = optimizer.param_groups[0]['lr']
    print(
        f'Epoch {epoch:3d}/{EPOCHS}'
        f'  recon={train_loss:.5f}'
        f'  val={val_loss:.5f}'
        f'  lr={lr_now:.2e}'
        f'  {elapsed:.1f}s'
        + ('  <-- best' if improved else '')
    )

mem_mb = torch.cuda.max_memory_allocated(device) / 1e6
avg_t  = sum(history['epoch_time']) / len(history['epoch_time'])
print(f'\nBest val recon loss : {best_val:.6f}  ->  {CKPT_DIR}/ae_best.pt')
print(f'Temps moyen / epoch : {avg_t:.1f}s')
print(f'Pic mémoire GPU     : {mem_mb:.0f} MB')

## 6. Loss curves

In [ ]:
epochs_x = range(1, len(history['train']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#111')

ax = axes[0]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['train'], color='#4fc3f7', label='train')
ax.plot(epochs_x, history['val'],   color='#ff8a65', label='val')
ax.set_title('Reconstruction MSE', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

ax = axes[1]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['epoch_time'], color='#a5d6a7', label='temps/epoch (s)')
ax.set_title('Temps par epoch', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

plt.tight_layout()
plt.savefig(f'{VIS_DIR}/ae_loss_colab.png', dpi=100, bbox_inches='tight', facecolor='#111')
plt.show()

## 7. Reconstruction visuelle

Compare les frames reconstruites par l'AE avec les frames réelles.  
Chaque colonne = un step de rollout (t+1 à t+5).

In [ ]:
import torch.nn.functional as F

model.eval()

# Prend un batch de validation
frames_batch, _ = next(iter(val_loader))
frames_batch = frames_batch.to(device)    # (B, T, 3, 64, 64)
B, T, C, H, W = frames_batch.shape

with torch.no_grad():
    pairs = model._make_pairs(frames_batch)                           # (B, T, 6, H, W)
    z     = model.encoder(pairs.reshape(B * T, 6, H, W)).view(B, T, EMBED_DIM)

n_show   = min(4, B)
horizons = list(range(1, ROLLOUT_K + 1))  # 1…5
n_cols   = len(horizons)

fig, axes = plt.subplots(n_show * 2, n_cols, figsize=(n_cols * 2, n_show * 4 + 0.5))
fig.patch.set_facecolor('#111')

for row, b in enumerate(range(n_show)):
    for col, k in enumerate(horizons):
        with torch.no_grad():
            z_roll = z[b:b+1, :T - k]    # (1, T-k, D)
            for _ in range(k):
                z_roll = model.predictor(z_roll)
            frame_pred = model.decoder(z_roll)  # (1, T-k, 3, H, W)

        # Premier frame disponible (t=0 prédit t+k)
        pred_img = frame_pred[0, 0].permute(1, 2, 0).cpu().numpy()
        real_img = frames_batch[b, k].permute(1, 2, 0).cpu().numpy()

        ax_pred = axes[row * 2,     col]
        ax_real = axes[row * 2 + 1, col]

        ax_pred.imshow(pred_img.clip(0, 1))
        ax_real.imshow(real_img.clip(0, 1))
        ax_pred.axis('off')
        ax_real.axis('off')

        if row == 0:
            ax_pred.set_title(f't+{k}', color='white', fontsize=9)
        if col == 0:
            ax_pred.set_ylabel(f'prédit', color='#4fc3f7', fontsize=8)
            ax_real.set_ylabel(f'réel',   color='#ff8a65', fontsize=8)

fig.suptitle('Reconstruction AE  (bleu=prédit, orange=réel)', color='white', fontsize=11)
plt.tight_layout()
plt.savefig(f'{VIS_DIR}/ae_reconstruction.png', dpi=120, bbox_inches='tight', facecolor='#111')
plt.show()

## 8. Download checkpoint

Downloads `ae_best.pt` to your browser.

In [ ]:
from google.colab import files
files.download(f'{CKPT_DIR}/ae_best.pt')